### Prepare 

In [1]:
from functions import *
print(check_jax_gpu())

Available GPUs:
Tesla P100-PCIE-16GB1: gpu
None


In [4]:
# necessary file path
settings_path = './settings_target/FZD7.json'
advanced_path = './settings_advanced/default_4stage_multimer.json'
filters_path = './settings_filters/default_filters.json'

# load settings json file
target_settings, advanced_settings, filters = load_json_settings(settings_path, filters_path, advanced_path)

# get settings file name
settings_file = os.path.basename(settings_path).split('.')[0]
filters_file = os.path.basename(filters_path).split('.')[0]
advanced_file = os.path.basename(advanced_path).split('.')[0]

print("Target settings:")
print(json.dumps(target_settings, indent=4))
print('Filter Used:', filters_file)
print('Advanced Used:', advanced_file)

Target settings:
{
    "design_path": "/hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7",
    "binder_name": "FZD7",
    "starting_pdb": "/hpf/projects/mtyers/ningrui/NXBindCraft/Targets/FZD7.pdb",
    "chains": "A",
    "target_hotspot_residues": "",
    "lengths": [
        50,
        100
    ],
    "number_of_final_designs": 20
}
Filter Used: default_filters
Advanced Used: default_4stage_multimer


In [ ]:
# AF2 model settings (which model to use)
design_models, prediction_models, multimer_validation = load_af2_models(advanced_settings["use_multimer_design"])

# set function paths (package settings)
bindcraft_folder = os.path.dirname('/hpf/projects/mtyers/ningrui/NXBindCraft/bindcraft.py') # main folder name, aka BindCraft...
advanced_settings["af_params_dir"] = bindcraft_folder
advanced_settings["dssp_path"] = os.path.join(bindcraft_folder, 'functions/dssp') # Define Secondary Structure of Proteins
advanced_settings["dalphaball_path"] = os.path.join(bindcraft_folder, 'functions/DAlphaBall.gcc') # computing solvent-accessible surface area (SASA) and buried surface area (BSA) in protein structures

# generate dicretories to store designs, stats and other output
design_paths = generate_directories(target_settings["design_path"])

# generate dataframes (possibility to store design stats)
trajectory_labels, design_labels, final_labels = generate_dataframe_labels()

# create csv file (path) to store stats 
trajectory_csv = os.path.join(target_settings["design_path"], 'trajectory_stats.csv')
mpnn_csv = os.path.join(target_settings["design_path"], 'mpnn_design_stats.csv')
final_csv = os.path.join(target_settings["design_path"], 'final_design_stats.csv')
failure_csv = os.path.join(target_settings["design_path"], 'failure_csv.csv')

# create csv file with label
create_dataframe(trajectory_csv, trajectory_labels)
create_dataframe(mpnn_csv, design_labels)
create_dataframe(final_csv, final_labels)
generate_filter_pass_csv(failure_csv, filters_path)


In [6]:
# initialise PyRosetta
pr.init(f'-ignore_unrecognized_res -ignore_zero_occupancy -mute all -holes:dalphaball {advanced_settings["dalphaball_path"]} -corrections::beta_nov16 true -relax:default_repeats 1')

# initialise counters
script_start_time = time.time()
trajectory_n = 1
accepted_designs = 0
rejected_designs = 0

┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2024 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python310.Release 2024.42+release.3366cf78a3df04339d1982e94531b77b098ddb99 2024-10-11T08:24:04] retrieved from: http://www.pyrosetta.org


### Start design loop!!!

In [ ]:
#### check if we have the target number of binders
#final_designs_reached = check_accepted_designs(design_paths, mpnn_csv, final_labels, final_csv, advanced_settings, target_settings, design_labels)
#
#if final_designs_reached: +++++++++++++++++++
#    # stop design loop execution
#    break

# random seed to vary different design
seed = int(np.random.randint(0, high=999999, size=1, dtype=int)[0])

# randomly find a binder length in the specified range [x,y]
samples = np.arange(min(target_settings["lengths"]), max(target_settings["lengths"]) + 1)
length = np.random.choice(samples)

# load desired helicity value --> dample different secondary structure contents
helicity_value = load_helicity(advanced_settings)

# generate design name & check if tranjectory was already run
design_name = target_settings["binder_name"] + "_l" + str(length) + "_s"+ str(seed) # l = length, s = seed
trajectory_dirs = ["Trajectory", "Trajectory/Relaxed", "Trajectory/LowConfidence", "Trajectory/Clashing"]
trajectory_exists = any(os.path.exists(os.path.join(design_paths[trajectory_dir], design_name + ".pdb")) for trajectory_dir in trajectory_dirs)


In [ ]:
# if not trajectory_exists: +++++++++++++++++++
print("Starting trajectory: "+design_name)

### begin binder hallucination 
# design_name: FZD7_l88_s105508
# length: 88
# seed: 105508
# helicity_value: -0.3
# design_models: [0, 1, 2, 3, 4]
# advanced_settings: <class 'dict'>
# design_paths: <class 'dict'>
# failure_csv: /hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7/failure_csv.csv
trajectory = binder_hallucination(design_name, target_settings["starting_pdb"], target_settings["chains"],
                                            target_settings["target_hotspot_residues"], length, seed, helicity_value,
                                            design_models, advanced_settings, design_paths, failure_csv)



{'Accepted': '/hpf/projects/mtyers/ningrui/BindCraft/Design/learningFZD7/Accepted',
 'Accepted/Ranked': '/hpf/projects/mtyers/ningrui/BindCraft/Design/learningFZD7/Accepted/Ranked',
 'Accepted/Animation': '/hpf/projects/mtyers/ningrui/BindCraft/Design/learningFZD7/Accepted/Animation',
 'Accepted/Plots': '/hpf/projects/mtyers/ningrui/BindCraft/Design/learningFZD7/Accepted/Plots',
 'Accepted/Pickle': '/hpf/projects/mtyers/ningrui/BindCraft/Design/learningFZD7/Accepted/Pickle',
 'Trajectory': '/hpf/projects/mtyers/ningrui/BindCraft/Design/learningFZD7/Trajectory',
 'Trajectory/Relaxed': '/hpf/projects/mtyers/ningrui/BindCraft/Design/learningFZD7/Trajectory/Relaxed',
 'Trajectory/Plots': '/hpf/projects/mtyers/ningrui/BindCraft/Design/learningFZD7/Trajectory/Plots',
 'Trajectory/Clashing': '/hpf/projects/mtyers/ningrui/BindCraft/Design/learningFZD7/Trajectory/Clashing',
 'Trajectory/LowConfidence': '/hpf/projects/mtyers/ningrui/BindCraft/Design/learningFZD7/Trajectory/LowConfidence',
 'Traj

In [16]:
print(design_name)
print(length)
print(seed)
print(helicity_value)
print(design_models)
print(type(advanced_settings))
print(design_paths)
print(failure_csv)

FZD7_l88_s105508
88
105508
-0.3
[0, 1, 2, 3, 4]
<class 'dict'>
{'Accepted': '/hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7/Accepted', 'Accepted/Ranked': '/hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7/Accepted/Ranked', 'Accepted/Animation': '/hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7/Accepted/Animation', 'Accepted/Plots': '/hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7/Accepted/Plots', 'Accepted/Pickle': '/hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7/Accepted/Pickle', 'Trajectory': '/hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7/Trajectory', 'Trajectory/Relaxed': '/hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7/Trajectory/Relaxed', 'Trajectory/Plots': '/hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7/Trajectory/Plots', 'Trajectory/Clashing': '/hpf/projects/mtyers/ningrui/NXBindCraft/Designs/learningFZD7/Trajectory/Clashing', 'Trajectory/LowConfidence': '/hpf/projec

In [17]:
nvidia-smi

NameError: name 'nvidia' is not defined

In [9]:
''' sdfsdfs '''

' sdfsdfs '